# Customer Churn Prediction

This notebook follows the complete workflow: business problem, data preparation, EDA, feature engineering, Decision Tree modeling, evaluation, interpretation, model saving, and API readiness.

In [ ]:
from pathlib import Path
import sys
import json
import joblib
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split

ROOT = Path.cwd().parent if Path.cwd().name == 'notebook' else Path.cwd()
sys.path.insert(0, str(ROOT))
from src.churn_pipeline import build_pipeline, get_feature_names, load_dataset, split_features_and_target
from src.train_model import CONFIGURATIONS
sns.set_theme(style='whitegrid', palette='deep')

## 1. Data Understanding and Preparation
The target is `Churn`. The customer identifier is excluded from modeling because it contains no generalizable customer behavior signal.

In [ ]:
data = load_dataset(ROOT / 'TelcoCustomerChurn.csv')
print('Shape:', data.shape)
display(data.head())
display(data.dtypes.to_frame('dtype'))
print('Duplicate rows:', data.duplicated().sum())
print('Missing values after numeric coercion:')
display(data.isna().sum().sort_values(ascending=False).head())
print('Target distribution:')
display(data['Churn'].value_counts().to_frame('count').assign(percent=lambda frame: frame['count'] / len(data) * 100))

`TotalCharges` contains blank values for new customers. These are converted to numeric missing values and imputed only after the train/test split inside the pipeline, preventing leakage.

In [ ]:
from src.churn_pipeline import ChurnFeatureEngineer
analysis_data = ChurnFeatureEngineer().fit_transform(data.drop(columns=['customerID', 'Churn']))
analysis_data['Churn'] = data['Churn'].values
analysis_data['tenure_group'] = analysis_data['tenure_group'].astype(str)
analysis_data['service_count'].describe()

## 2. Exploratory Data Analysis
The plots below focus on actionable customer and service characteristics. Each section records a business interpretation rather than treating association as causation.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.countplot(data=analysis_data, x='Churn', ax=axes[0])
axes[0].set_title('Churn distribution')
sns.countplot(data=analysis_data, x='Contract', hue='Churn', ax=axes[1])
axes[1].set_title('Churn by contract type')
axes[1].tick_params(axis='x', rotation=20)
plt.tight_layout()
# Insight: churn is imbalanced, and month-to-month customers visibly represent the highest-risk contract group.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.barplot(data=analysis_data, x='InternetService', y='Churn', estimator=lambda values: np.mean(values == 'Yes'), ax=axes[0])
axes[0].set_title('Churn rate by internet service')
sns.barplot(data=analysis_data, x='PaymentMethod', y='Churn', estimator=lambda values: np.mean(values == 'Yes'), ax=axes[1])
axes[1].set_title('Churn rate by payment method')
axes[1].tick_params(axis='x', rotation=25)
plt.tight_layout()
# Insight: service type and payment experience identify segments for targeted retention journeys.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.barplot(data=analysis_data, x='tenure_group', y='Churn', estimator=lambda values: np.mean(values == 'Yes'), ax=axes[0])
axes[0].set_title('Churn rate by tenure group')
axes[0].tick_params(axis='x', rotation=20)
sns.boxplot(data=analysis_data, x='Churn', y='MonthlyCharges', ax=axes[1])
axes[1].set_title('Monthly charges by churn outcome')
plt.tight_layout()
# Insight: early-tenure customers and higher monthly charges are useful signals for onboarding and value interventions.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(data=analysis_data, x='tenure', hue='Churn', bins=20, kde=True, element='step', ax=axes[0])
axes[0].set_title('Tenure distribution')
sns.scatterplot(data=analysis_data, x='MonthlyCharges', y='TotalCharges', hue='Churn', alpha=0.5, ax=axes[1])
axes[1].set_title('Monthly and total charges')
plt.tight_layout()
# Insight: total charges are strongly related to tenure and monthly charges, so both should be interpreted together.

## 3. Feature Engineering
`tenure_group` converts lifecycle duration into business-readable bands. `service_count` summarizes the number of subscribed services marked Yes, providing a compact engagement measure. Both are calculated inside the fitted pipeline for new API records too.

## 4. Decision Tree Models
The split is stratified and fixed at 70:30 with seed 42. Preprocessing is fitted using training data only.

In [ ]:
features, target = split_features_and_target(data)
X_train, X_test, y_train, y_test = train_test_split(features, target, test_size=0.30, random_state=42, stratify=target)
models = {}
results = {}
for name, configuration in CONFIGURATIONS.items():
    fitted_model = build_pipeline(**configuration)
    fitted_model.fit(X_train, y_train)
    prediction = fitted_model.predict(X_test)
    models[name] = fitted_model
    results[name] = {
        'accuracy': accuracy_score(y_test, prediction),
        'precision': precision_score(y_test, prediction, zero_division=0),
        'recall': recall_score(y_test, prediction, zero_division=0),
        'f1_score': f1_score(y_test, prediction, zero_division=0),
    }
results_frame = pd.DataFrame(results).T.sort_values('f1_score', ascending=False)
display(results_frame)

The final model prioritizes recall, then F1 score and precision. For retention outreach, recall is generally more valuable because a false negative misses a customer who could potentially be retained; the business can later tune the probability threshold based on contact capacity.

In [ ]:
selected_name = max(results, key=lambda name: (results[name]['recall'], results[name]['f1_score'], results[name]['precision']))
selected_model = models[selected_name]
selected_prediction = selected_model.predict(X_test)
print('Selected model:', selected_name)
print('Confusion matrix:', confusion_matrix(y_test, selected_prediction))
ConfusionMatrixDisplay.from_predictions(y_test, selected_prediction, display_labels=['No', 'Yes'], cmap='Blues')
plt.title('Final model confusion matrix')
plt.show()

## 5. Model Interpretation
Decision trees are interpretable through their split-based feature importances. One-hot encoded service and contract categories are reported individually.

In [ ]:
importance = pd.DataFrame({
    'feature': get_feature_names(selected_model),
    'importance': selected_model.named_steps['classifier'].feature_importances_
}).sort_values('importance', ascending=False)
display(importance.head(15))
sns.barplot(data=importance.head(10), x='importance', y='feature')
plt.title('Top feature importances')
plt.tight_layout()
plt.show()

Typical operational actions include reviewing month-to-month customers, early-tenure onboarding, billing/payment friction, and high-value customers with elevated service charges. Feature importance indicates predictive association, not that changing a feature will necessarily cause retention.

In [ ]:
model_path = ROOT / 'model' / 'churn_model.pkl'
model_path.parent.mkdir(exist_ok=True)
joblib.dump(selected_model, model_path)
print('Saved:', model_path)

## 6. API Readiness
The Flask API loads the saved pipeline and applies the same feature engineering and preprocessing to each JSON request. See `app.py`, `sample_request.json`, and the README for execution instructions.